## Kanana-1.5-V 3B Unified RAG Champion

목표는 기존 최고점 **44.8395**를 넘어 47점 이상, 가능하면 50점 이상을
탐색하는 것입니다. 점수 달성을 보장하는 노트북은 아니며, 검증 지표와
리더보드 결과로 최종 판단해야 합니다.

핵심 변경점:

- 팀 AIVQA 파이프라인: Shared LoRA 2 epoch + MC/SA/LA 유형별 best Adapter
- 2,500건 통합 DB의 KURE 텍스트 검색 + 가용할 때만 CLIP 이미지 검색
- 실제 RAG 이미지가 없으면 README 규칙대로 text-only Qdrant로 자동 전환
- 검색 결과를 강제로 넣지 않는 **고신뢰 selective RAG**
- 유형별 RAG 설명 길이 제한: MC 700 / SA 550 / LA 1,200자
- 유형별 학습 예산: MC 3 / SA 5 / LA 3 epoch, 매 epoch 검증 최고점 저장
- SA는 자유 생성 후 길이 위반 시에만 음절·어절 hard retry
- 검색 cache, Qdrant DB, Adapter, 제출 파일을 Drive에 저장

Colab L4 또는 A100 GPU에서 위에서부터 실행하세요. 설치 셀 뒤 런타임이
자동 재시작되면 다시 `런타임 > 모두 실행`을 누르세요.

### 0. 환경 설치

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


marker = Path("/content/.aivqa_unified_champion_env_v1")
if not marker.exists():
    def pip_install(*packages):
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", *packages]
        )

    subprocess.call(
        [
            sys.executable, "-m", "pip", "uninstall", "-q", "-y",
            "gradio", "gradio_client", "torchao", "torchaudio",
        ]
    )
    pip_install(
        "transformers>=4.57.0,<5.0.0",
        "accelerate>=1.0.0,<2.0.0",
        "peft>=0.17.0,<1.0.0",
        "qdrant-client>=1.13.0,<2.0.0",
        "sentence-transformers>=5.0.0,<6.0.0",
        "bitsandbytes>=0.46.0,<1.0.0",
        "timm>=1.0.0,<2.0.0",
        "einops>=0.8.0,<1.0.0",
        "omegaconf>=2.3.0,<3.0.0",
        "safetensors>=0.4.3,<1.0.0",
        "pandas>=2.0.0,<3.0.0",
        "tqdm>=4.66.0,<5.0.0",
    )
    pip_install(
        "--no-cache-dir", "--force-reinstall", "Pillow>=12.0.0,<13.0.0"
    )
    marker.write_text("ready", encoding="utf-8")
    print("설치 완료. 런타임을 재시작합니다.")
    print("재연결 후 첫 셀부터 다시 모두 실행하세요.")
    os.kill(os.getpid(), 9)

import importlib.metadata
import PIL
import transformers

print("Python:", sys.version.split()[0])
print("Pillow:", PIL.__version__)
print("Transformers:", transformers.__version__)
print("Qdrant client:", importlib.metadata.version("qdrant-client"))

### 1. Google Drive 연결 및 실험 설정

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import argparse
import copy
import gc
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import sys
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm


DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/CV_korea")
DRIVE_ZIP_DIR = DRIVE_PROJECT_DIR / "data"
RAG_BUNDLE_DIR = DRIVE_PROJECT_DIR / "unified_rag_db"
RAG_JSONL_PATH = RAG_BUNDLE_DIR / "unified_rag.jsonl"

REPO_URL = "https://github.com/kjh0902/AIVQA.git"
REPO_COMMIT = "e961b1f3fbb5d77ef4414e7644e63b913177c827"
MODEL_ID = "kakaocorp/kanana-1.5-v-3b-instruct"
KANANA_REVISION = "2e00ef13ccec2e99459a8eada18a1bfd05bff44b"

RUN_NAME = "kanana15v_3b_unified_rag_champion_v1"
COLAB_ROOT = Path("/content/aivqa_unified_champion")
REPO_DIR = COLAB_ROOT / "AIVQA"
EXTRACT_DIR = COLAB_ROOT / "extracted"
DATASET_ROOT = COLAB_ROOT / "dataset_root"
LOCAL_RAG_DIR = COLAB_ROOT / "rag_bundle"
LOCAL_RAG_JSONL = LOCAL_RAG_DIR / "unified_rag.jsonl"
LOCAL_QDRANT_DIR = COLAB_ROOT / "qdrant_storage"
MODEL_CACHE_DIR = COLAB_ROOT / "model_cache"
RUN_DIR = DRIVE_PROJECT_DIR / "outputs" / RUN_NAME

# Champion 전체 학습 노트북이므로 True로 유지합니다.
DO_TRAIN = True
REBUILD_QDRANT = False

SHARED_EPOCHS = 2
TYPE_EPOCHS_BY_FORM = {"MC": 3, "SA": 5, "LA": 3}
SHARED_LEARNING_RATE = 5e-5
TYPE_LEARNING_RATE = 2e-5
GRADIENT_ACCUMULATION_STEPS = 8
MAX_LENGTH = 4096
MIN_PIXELS = 100 * 28 * 28
MAX_PIXELS = 600 * 28 * 28
SEED = 42

# 점수가 서로 다른 KURE와 CLIP을 단순 합산하지 않습니다.
RETRIEVAL_BASE_THRESHOLD = 0.80
DUAL_TEXT_THRESHOLD = 0.82
DUAL_IMAGE_THRESHOLD = 0.82
STRONG_TEXT_THRESHOLD = 0.93
STRONG_IMAGE_THRESHOLD = 0.965
RAG_TOP_K = 1
RAG_CONTEXT_LIMITS = {"MC": 700, "SA": 550, "LA": 1200}

for path in (
    DRIVE_PROJECT_DIR, COLAB_ROOT, EXTRACT_DIR, DATASET_ROOT,
    LOCAL_RAG_DIR, MODEL_CACHE_DIR, RUN_DIR,
):
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("대회 ZIP:", DRIVE_ZIP_DIR)
print("통합 RAG:", RAG_JSONL_PATH)
print("실험 결과:", RUN_DIR)
print("학습:", SHARED_EPOCHS, "+", TYPE_EPOCHS_BY_FORM)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임 유형을 GPU로 변경하세요.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
dtype_name = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print("GPU:", gpu_name)
print(f"VRAM: {gpu_mem_gb:.1f} GiB")
print("학습 dtype:", dtype_name)
if DO_TRAIN and gpu_mem_gb < 20:
    raise RuntimeError("일반 LoRA 학습은 L4/A100급(20GB 이상) GPU를 권장합니다.")

### 2. 팀 AIVQA 코드 고정 및 대회 데이터 준비

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin"])

subprocess.check_call(
    ["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_COMMIT]
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == REPO_COMMIT, (actual_commit, REPO_COMMIT)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("AIVQA commit:", actual_commit)

In [ ]:
EXPECTED_ARCHIVES = [
    "한국문화 멀티모달 질의응답.zip",
    "train.zip",
    "validation.zip",
    "test.zip",
]


def normalized_name(text):
    return unicodedata.normalize("NFC", str(text))


def resolve_archive(directory, expected_name):
    expected_nfc = normalized_name(expected_name)
    matches = [
        path for path in directory.glob("*.zip")
        if normalized_name(path.name) == expected_nfc
    ]
    if len(matches) != 1:
        found = [path.name for path in directory.glob("*.zip")]
        raise FileNotFoundError(
            f"{expected_name!r}을 찾지 못했습니다. 현재 ZIP: {found}"
        )
    return matches[0]


archive_paths = [
    resolve_archive(DRIVE_ZIP_DIR, name) for name in EXPECTED_ARCHIVES
]
archive_state = {
    path.name: {"size": path.stat().st_size, "mtime": path.stat().st_mtime_ns}
    for path in archive_paths
}
state_path = EXTRACT_DIR / ".archive_state.json"
previous = (
    json.loads(state_path.read_text(encoding="utf-8"))
    if state_path.is_file() else None
)
if previous == archive_state:
    print("동일한 ZIP 압축 해제 결과를 재사용합니다.")
else:
    for archive_path in archive_paths:
        print("압축 해제:", archive_path.name)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(EXTRACT_DIR)
    state_path.write_text(
        json.dumps(archive_state, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

In [ ]:
def find_split_json(split):
    matches = [
        path for path in EXTRACT_DIR.rglob("*.json")
        if normalized_name(path.stem).endswith(f"_{split}")
    ]
    if len(matches) != 1:
        raise RuntimeError(f"{split} JSON 후보가 1개가 아닙니다: {matches}")
    return matches[0]


def read_json(path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def find_split_image_dir(split, rows):
    sample_names = [row["model_input"]["image_name"] for row in rows[:10]]
    scores = Counter()
    for image_name in sample_names:
        for candidate in EXTRACT_DIR.rglob(image_name):
            if candidate.is_file():
                scores[candidate.parent] += 1
    if not scores:
        raise FileNotFoundError(f"{split} 이미지 디렉터리를 찾지 못했습니다.")
    best_dir, score = scores.most_common(1)[0]
    if score < min(3, len(sample_names)):
        raise RuntimeError(f"{split} 이미지 경로 탐색이 불안정합니다: {scores}")
    return best_dir


json_paths = {
    split: find_split_json(split)
    for split in ("train", "validation", "test")
}
records = {split: read_json(path) for split, path in json_paths.items()}
image_dirs = {
    split: find_split_image_dir(split, records[split])
    for split in records
}

# AIVQA Dataset이 dataset_root/split/image_name으로 읽도록 링크합니다.
for split, image_dir in image_dirs.items():
    target = DATASET_ROOT / split
    if target.is_symlink() and target.resolve() != image_dir.resolve():
        target.unlink()
    elif target.exists() and not target.is_symlink():
        raise RuntimeError(f"링크 대상 경로가 이미 디렉터리입니다: {target}")
    if not target.exists():
        target.symlink_to(image_dir, target_is_directory=True)

expected_counts = {"train": 1000, "validation": 200, "test": 800}
rows = []
for split, split_rows in records.items():
    missing = [
        row["model_input"]["image_name"] for row in split_rows
        if not (image_dirs[split] / row["model_input"]["image_name"]).is_file()
    ]
    forms = Counter(row["metadata"]["question_form"] for row in split_rows)
    assert len(split_rows) == expected_counts[split], (split, len(split_rows))
    assert not missing, (split, missing[:5])
    rows.append({
        "split": split,
        "rows": len(split_rows),
        "MC": forms["MC"], "SA": forms["SA"], "LA": forms["LA"],
        "missing_images": len(missing),
        "image_dir": str(image_dirs[split]),
    })
display(pd.DataFrame(rows))

### 3. 통합 RAG JSONL 검증 및 선택적 이미지 복사

In [ ]:
if not RAG_JSONL_PATH.is_file():
    candidates = list(DRIVE_PROJECT_DIR.rglob("unified_rag.jsonl"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Drive의 CV_korea/unified_rag_db/unified_rag.jsonl와 "
            "그 옆 images/ 폴더를 준비하세요. 발견 후보: " + str(candidates)
        )
    RAG_JSONL_PATH = candidates[0]
    RAG_BUNDLE_DIR = RAG_JSONL_PATH.parent


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


rag_rows = []
invalid_lines = []
with RAG_JSONL_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        try:
            value = json.loads(line)
        except json.JSONDecodeError as error:
            invalid_lines.append((line_number, str(error)))
            continue
        if isinstance(value, dict):
            rag_rows.append(value)

required = {
    "doc_id", "source", "title", "search_terms", "description", "image_path"
}
assert not invalid_lines, invalid_lines[:5]
assert len(rag_rows) > 0
assert all(required.issubset(row) for row in rag_rows)
assert len({str(row["doc_id"]) for row in rag_rows}) == len(rag_rows)

source_counts = Counter(str(row["source"]) for row in rag_rows)
declared = 0
copied = 0
missing = []
normalized_rows = []
for row in tqdm(rag_rows, desc="RAG 이미지 로컬 복사"):
    row = dict(row)
    raw_paths = row.get("image_path") or []
    if isinstance(raw_paths, str):
        raw_paths = [raw_paths]
    normalized_paths = []
    for image_index, raw in enumerate(raw_paths):
        declared += 1
        raw_path = Path(str(raw)).expanduser()
        source_path = (
            raw_path if raw_path.is_absolute()
            else RAG_JSONL_PATH.parent / raw_path
        )
        if not source_path.is_file():
            missing.append(str(raw))
            continue
        if raw_path.is_absolute():
            suffix = source_path.suffix.lower() or ".jpg"
            relative = Path("images") / "imported" / (
                f"{row['doc_id']}_{image_index}{suffix}"
            )
        else:
            relative = raw_path
        destination = LOCAL_RAG_DIR / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if (
            not destination.is_file()
            or destination.stat().st_size != source_path.stat().st_size
        ):
            shutil.copy2(source_path, destination)
        normalized_paths.append(relative.as_posix())
        copied += 1
    row["image_path"] = normalized_paths
    normalized_rows.append(row)

coverage = copied / declared if declared else 1.0
RAG_BUILD_MODE = "text_image" if copied else "text_only"
if missing:
    print(
        f"실제 파일이 없는 image_path {len(missing)}개는 README 규칙대로 "
        "image vector만 생략합니다."
    )

with LOCAL_RAG_JSONL.open("w", encoding="utf-8") as file:
    for row in normalized_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

RAG_SHA256 = file_sha256(LOCAL_RAG_JSONL)
RETRIEVAL_SIGNATURE = {
    "rag_sha256": RAG_SHA256,
    "repo_commit": REPO_COMMIT,
    "thresholds": {
        "base": RETRIEVAL_BASE_THRESHOLD,
        "dual_text": DUAL_TEXT_THRESHOLD,
        "dual_image": DUAL_IMAGE_THRESHOLD,
        "strong_text": STRONG_TEXT_THRESHOLD,
        "strong_image": STRONG_IMAGE_THRESHOLD,
    },
    "top_k": RAG_TOP_K,
    "context_limits": RAG_CONTEXT_LIMITS,
}
display(pd.DataFrame(source_counts.items(), columns=["source", "documents"]))
print("RAG 문서:", len(normalized_rows))
print("설명 없음:", sum(not str(row.get("description", "")).strip() for row in normalized_rows))
print(f"이미지 복사: {copied}/{declared} ({coverage:.1%})")
print("Qdrant 구축 모드:", RAG_BUILD_MODE)
print("RAG SHA-256:", RAG_SHA256)

### 4. KURE 중심 Qdrant DB 생성 또는 Drive cache 복원

In [ ]:
qdrant_cache_dir = DRIVE_PROJECT_DIR / "qdrant_cache"
qdrant_cache_dir.mkdir(parents=True, exist_ok=True)
QDRANT_ARCHIVE = qdrant_cache_dir / (
    f"unified_{RAG_SHA256[:12]}_{REPO_COMMIT[:8]}_qdrant.zip"
)
QDRANT_COLLECTION = "aivqa_unified_rag"

if LOCAL_QDRANT_DIR.exists():
    shutil.rmtree(LOCAL_QDRANT_DIR)

if QDRANT_ARCHIVE.is_file() and not REBUILD_QDRANT:
    print("Drive의 동일 DB Qdrant cache를 복원합니다:", QDRANT_ARCHIVE)
    LOCAL_QDRANT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(QDRANT_ARCHIVE) as archive:
        archive.extractall(LOCAL_QDRANT_DIR)
else:
    print(
        "2,500건의 KURE text vector를 생성합니다. "
        "실제 이미지가 있는 문서만 CLIP vector도 함께 저장합니다."
    )
    command = [
        sys.executable,
        str(REPO_DIR / "rag_db" / "build_qdrant.py"),
        "--input", str(LOCAL_RAG_JSONL),
        "--qdrant-path", str(LOCAL_QDRANT_DIR),
        "--collection", QDRANT_COLLECTION,
        "--model-cache", str(MODEL_CACHE_DIR),
        "--device", "cuda",
        "--batch-size", "64",
        "--text-batch-size", "64",
        "--image-batch-size", "32",
        "--recreate",
    ]
    subprocess.check_call(command, cwd=str(REPO_DIR))

    local_archive_base = COLAB_ROOT / f"unified_{RAG_SHA256[:12]}_qdrant"
    local_zip = Path(
        shutil.make_archive(
            str(local_archive_base), "zip", root_dir=LOCAL_QDRANT_DIR
        )
    )
    shutil.copy2(local_zip, QDRANT_ARCHIVE)
    print("다음 실행용 cache 저장:", QDRANT_ARCHIVE)

assert LOCAL_QDRANT_DIR.is_dir()
print("Qdrant 로컬 경로:", LOCAL_QDRANT_DIR)

### 5. 선택적 RAG·Kanana 안전 로더·SA 후처리 적용

In [ ]:
"""Runtime patches embedded in the unified-RAG Kanana champion Colab notebook.

This module is not imported by the local project.  The notebook builder embeds its
source after cloning the pinned AIVQA repository in Colab.
"""

from __future__ import annotations

import copy
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import LogitsProcessor, LogitsProcessorList

import aivqa.data as aivqa_data
import aivqa.metrics as aivqa_metrics
import rag_db.augmentation as rag_augmentation
import rag_db.infer_with_rag as rag_infer
import rag_db.prompts as rag_prompts
import run_rag_pipeline as rag_pipeline
import train_lora
import type_adapters.modeling as type_modeling
import type_adapters.train as type_train


# ---------------------------------------------------------------------------
# 1. Colab에서 검증된 Kanana revision + 내부 vision SDPA 강제 로더
# ---------------------------------------------------------------------------


def _kanana_model_class():
    from transformers.dynamic_module_utils import get_class_from_dynamic_module

    model_class = get_class_from_dynamic_module(
        "modeling.KananaVForConditionalGeneration",
        MODEL_ID,
        revision=KANANA_REVISION,
    )
    vision_class = model_class.__init__.__globals__["CustomQwen2VLVE"]
    if not getattr(vision_class, "_champion_sdpa_patched", False):
        original_from_config = vision_class._from_config.__func__

        @classmethod
        def from_config_with_sdpa(cls, config, **kwargs):
            kwargs["attn_implementation"] = "sdpa"
            return original_from_config(cls, config, **kwargs)

        vision_class._from_config = from_config_with_sdpa
        vision_class._champion_sdpa_patched = True
    return model_class


def _processor_for(args):
    from transformers import AutoProcessor

    processor = AutoProcessor.from_pretrained(
        args.model_id,
        revision=KANANA_REVISION,
        trust_remote_code=True,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    train_lora.configure_image_pixel_limits(
        processor, args.min_pixels, args.max_pixels
    )
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    return processor


def _base_model_for(args, dtype):
    from transformers import BitsAndBytesConfig

    model_kwargs = {
        "revision": KANANA_REVISION,
        "dtype": dtype,
        "device_map": {"": torch.cuda.current_device()},
        "low_cpu_mem_usage": True,
        "attn_implementation": "sdpa",
        "cache_dir": str(MODEL_CACHE_DIR),
    }
    if args.load_in_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
    return _kanana_model_class().from_pretrained(args.model_id, **model_kwargs)


def safe_build_model_and_processor(args):
    from peft import LoraConfig, TaskType, get_peft_model

    if not torch.cuda.is_available():
        raise RuntimeError("Kanana 학습에는 CUDA GPU가 필요합니다.")
    dtype = getattr(torch, args.dtype)
    processor = _processor_for(args)
    model = _base_model_for(args, dtype)
    model.requires_grad_(False)
    model.config.use_cache = False
    model.language_model.config.use_cache = False
    llm = train_lora._prepare_llm_for_training(
        model,
        load_in_4bit=args.load_in_4bit,
        gradient_checkpointing=args.gradient_checkpointing,
    )
    target_modules = train_lora.find_adapter_target_modules(
        name for name, _ in llm.named_modules()
    )
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        target_modules=target_modules,
        bias="none",
    )
    peft_llm = get_peft_model(llm, config)
    if hasattr(llm, "_require_grads_hook"):
        llm.disable_input_require_grads()
    model.language_model = peft_llm
    train_lora._verify_only_llm_adapters_are_trainable(model)
    model.language_model.print_trainable_parameters()
    return model, processor, dtype


def safe_load_base_model_and_processor(args, *, for_training):
    if not torch.cuda.is_available():
        raise RuntimeError("Kanana 실행에는 CUDA GPU가 필요합니다.")
    dtype = getattr(torch, args.dtype)
    processor = _processor_for(args)
    model = _base_model_for(args, dtype)
    model.requires_grad_(False)
    if for_training:
        model.config.use_cache = False
        model.language_model.config.use_cache = False
        model.language_model = train_lora._prepare_llm_for_training(
            model,
            load_in_4bit=args.load_in_4bit,
            gradient_checkpointing=args.gradient_checkpointing,
        )
    else:
        model.eval()
    return model, processor, dtype


rag_pipeline.build_model_and_processor = safe_build_model_and_processor
rag_pipeline.load_base_model_and_processor = safe_load_base_model_and_processor
rag_infer.load_base_model_and_processor = safe_load_base_model_and_processor
type_modeling.load_base_model_and_processor = safe_load_base_model_and_processor
type_train.load_base_model_and_processor = safe_load_base_model_and_processor


# ---------------------------------------------------------------------------
# 2. KURE/CLIP score를 그대로 더하지 않는 고신뢰 selective retrieval
#    JSONL에 실제 이미지 파일이 없으면 README 설계대로 text-only로 동작합니다.
# ---------------------------------------------------------------------------


def safe_retriever_init(
    self,
    client,
    collection_name,
    encoders,
    threshold,
    retrieval_page_size,
    local_image_search=False,
):
    self.client = client
    self.collection_name = collection_name
    self.encoders = encoders
    self.threshold = threshold
    self.retrieval_page_size = retrieval_page_size
    self.title_index = rag_infer.load_title_index(client, collection_name)
    self.local_image_index = None
    self.image_search_enabled = False
    if local_image_search:
        try:
            self.local_image_index = rag_infer.LocalImageIndex.load(
                client, collection_name
            )
            self.image_search_enabled = True
        except ValueError as error:
            if "no image vectors" not in str(error).lower():
                raise
            print(
                "Qdrant에 image vector가 없어 KURE text-only RAG로 전환합니다."
            )
    else:
        # Server mode에서는 native image query가 가능하다고 가정합니다.
        self.image_search_enabled = True


rag_infer.QdrantRetriever.__init__ = safe_retriever_init


def _candidate_gate(candidate):
    text_score = float(candidate.text_score)
    image_score = float(candidate.image_score)
    exact_title = text_score >= 1.99
    dual = text_score >= DUAL_TEXT_THRESHOLD and image_score >= DUAL_IMAGE_THRESHOLD
    strong_text = text_score >= STRONG_TEXT_THRESHOLD
    near_duplicate_image = image_score >= STRONG_IMAGE_THRESHOLD

    if exact_title:
        return "exact_title", 1.20 + min(image_score, 1.0) * 0.20
    if near_duplicate_image:
        return "near_duplicate_image", 1.10 + image_score * 0.10
    if dual:
        return "dual_channel", 0.55 * text_score + 0.45 * image_score + 0.10
    if strong_text:
        return "strong_text", text_score
    return None


def selective_retrieve(self, search_terms: Sequence[str], image):
    candidates = {}
    for term in search_terms:
        exact_doc_ids = self.title_index.get(rag_infer.normalize_exact_text(term), [])
        if exact_doc_ids:
            for payload in self._retrieve_payloads(exact_doc_ids):
                self._merge(candidates, payload, text_score=2.0)
            continue
        for point in self._query_all(
            self.encoders.embed_text(term), rag_infer.TEXT_VECTOR_NAME
        ):
            self._merge(
                candidates,
                rag_infer.validate_payload(point.payload),
                text_score=float(point.score),
            )

    if self.image_search_enabled:
        image_vector = self.encoders.embed_image(image)
        if self.local_image_index is not None:
            image_hits = self.local_image_index.search(image_vector, self.threshold)
            for payload, score in image_hits:
                self._merge(candidates, payload, image_score=score)
        else:
            for point in self._query_all(
                [image_vector], rag_infer.IMAGE_VECTOR_NAME, require_vector=True
            ):
                self._merge(
                    candidates,
                    rag_infer.validate_payload(point.payload),
                    image_score=float(point.score),
                )

    accepted = []
    for candidate in candidates.values():
        gate = _candidate_gate(candidate)
        if gate is None:
            continue
        reason, confidence = gate
        payload = dict(candidate.payload)
        payload["_retrieval"] = {
            "reason": reason,
            "confidence": float(confidence),
            "text_score": float(candidate.text_score),
            "image_score": float(candidate.image_score),
        }
        accepted.append(
            (
                confidence,
                rag_prompts.Candidate(
                    doc_id=candidate.doc_id,
                    payload=payload,
                    text_score=float(candidate.text_score),
                    image_score=float(candidate.image_score),
                ),
            )
        )
    accepted.sort(key=lambda item: (-item[0], item[1].doc_id))
    return [candidate for _, candidate in accepted[:RAG_TOP_K]]


rag_infer.QdrantRetriever.retrieve = selective_retrieve


# ---------------------------------------------------------------------------
# 3. 질문 관련 문장만 넣는 짧은 top-1 RAG prompt
# ---------------------------------------------------------------------------


_GENERIC_TERMS = {
    "사진",
    "이미지",
    "그림",
    "대상",
    "설명",
    "내용",
    "이름",
    "무엇",
    "해당",
    "다음",
    "우리나라",
    "한국",
    "답하시오",
    "고르시오",
}


def _clean_rag_text(text):
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", str(text))
    text = re.sub(r"\r\n?", "\n", text)
    return re.sub(r"[ \t]+", " ", text).strip()


def _query_terms(question, options):
    text = " ".join([str(question), *map(str, options)])
    terms = re.findall(r"[0-9A-Za-z가-힣]{2,}", text)
    return [term for term in terms if term not in _GENERIC_TERMS]


def _relevant_excerpt(description, question, options, title, limit):
    description = _clean_rag_text(description)
    if len(description) <= limit:
        return description
    pieces = [
        piece.strip()
        for piece in re.split(r"\n+|(?<=[.!?。])\s+", description)
        if piece.strip()
    ]
    if not pieces:
        return description[:limit].rstrip()
    terms = _query_terms(question, options)
    title_terms = re.findall(r"[0-9A-Za-z가-힣]{2,}", str(title))
    scored = []
    for index, piece in enumerate(pieces):
        compact = piece.casefold()
        score = sum(2.0 for term in title_terms if term.casefold() in compact)
        score += sum(1.0 for term in terms if term.casefold() in compact)
        if index == 0:
            score += 0.25
        scored.append((score, index, piece))
    selected = sorted(scored, key=lambda item: (-item[0], item[1]))[:4]
    selected.sort(key=lambda item: item[1])
    output = " ".join(piece for _, _, piece in selected).strip()
    return output[:limit].rstrip()


def champion_build_answer_feature(
    sample,
    question,
    options,
    candidates,
    max_rag_chars=None,
):
    question_form = sample["question_form"]
    system_prompt = (
        f"{aivqa_data.SYSTEM_PROMPT}\n\n"
        f"{aivqa_data.QUESTION_FORM_INSTRUCTIONS[question_form]}\n\n"
        "참고자료는 검색 신뢰 기준을 통과한 경우에만 제공됩니다. "
        "그래도 이미지와 질문에 맞지 않으면 참고자료를 무시하세요."
    )
    parts = [aivqa_data.format_question(question_form, question, options)]
    if candidates:
        candidate = candidates[0]
        payload = candidate.payload
        limit = RAG_CONTEXT_LIMITS[question_form]
        excerpt = _relevant_excerpt(
            payload.get("description", ""),
            question,
            options,
            payload.get("title", ""),
            limit,
        )
        if excerpt:
            retrieval = payload.get("_retrieval", {})
            parts.append(
                "[검색 신뢰 기준을 통과한 참고자료]\n"
                f"제목: {payload.get('title', '')}\n"
                f"출처: {payload.get('source', '')}\n"
                f"관련 설명: {excerpt}\n"
                f"검색 근거: {retrieval.get('reason', 'verified')}"
            )
    return {
        "conversation": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "<image>"},
            {"role": "user", "content": "\n\n".join(parts)},
        ],
        "image": sample["image"],
    }


rag_prompts.build_answer_feature = champion_build_answer_feature
rag_augmentation.build_answer_feature = champion_build_answer_feature
rag_infer.build_answer_feature = champion_build_answer_feature


# ---------------------------------------------------------------------------
# 4. 검색 cache를 20건마다 Drive에 저장하고 재실행 시 재사용
# ---------------------------------------------------------------------------


def _candidate_to_dict(candidate):
    return {
        "doc_id": candidate.doc_id,
        "text_score": float(candidate.text_score),
        "image_score": float(candidate.image_score),
        "payload": candidate.payload,
    }


def _candidate_from_dict(value):
    return rag_prompts.Candidate(
        doc_id=str(value["doc_id"]),
        payload=dict(value["payload"]),
        text_score=float(value.get("text_score", 0.0)),
        image_score=float(value.get("image_score", 0.0)),
    )


def _atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(path)


def resumable_retrieve_dataset_candidates(
    model,
    processor,
    dataset,
    retriever,
    *,
    max_length,
    search_max_new_tokens,
    dtype,
    description,
    cache_path=None,
):
    cache_path = Path(cache_path) if cache_path is not None else None
    cache_rows = []
    if cache_path is not None and cache_path.is_file():
        try:
            cached = json.loads(cache_path.read_text(encoding="utf-8"))
            if cached.get("signature") == RETRIEVAL_SIGNATURE:
                cache_rows = list(cached.get("rows", []))
        except Exception as error:
            print("검색 cache를 무시합니다:", error)

    all_candidates = []
    completed_since_save = 0
    for index in tqdm(range(len(dataset)), desc=description, unit="sample"):
        sample = dataset[index]
        question_id = str(sample.get("question_id", index))
        cached_row = cache_rows[index] if index < len(cache_rows) else None
        if (
            isinstance(cached_row, dict)
            and cached_row.get("question_id") == question_id
        ):
            all_candidates.append(
                [_candidate_from_dict(item) for item in cached_row.get("candidates", [])]
            )
            continue

        search_output = rag_infer.generate_one(
            model,
            processor,
            rag_prompts.build_search_feature(sample, sample["question"]),
            max_length,
            search_max_new_tokens,
            dtype,
        )
        search_terms = rag_infer.parse_search_terms(search_output)
        candidates = retriever.retrieve(search_terms, sample["image"])
        row = {
            "question_id": question_id,
            "search_terms": search_terms,
            "candidates": [_candidate_to_dict(item) for item in candidates],
        }
        if index < len(cache_rows):
            cache_rows[index] = row
        else:
            cache_rows.append(row)
        all_candidates.append(candidates)
        completed_since_save += 1
        if cache_path is not None and completed_since_save >= 20:
            _atomic_json(
                cache_path,
                {"signature": RETRIEVAL_SIGNATURE, "rows": cache_rows},
            )
            completed_since_save = 0

    if cache_path is not None:
        _atomic_json(
            cache_path,
            {"signature": RETRIEVAL_SIGNATURE, "rows": cache_rows},
        )
    return all_candidates


rag_augmentation.retrieve_dataset_candidates = resumable_retrieve_dataset_candidates
rag_pipeline.retrieve_dataset_candidates = resumable_retrieve_dataset_candidates


def fixed_run_output_dir(_output_root):
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    return RUN_DIR


rag_pipeline.create_run_output_dir = fixed_run_output_dir


# ---------------------------------------------------------------------------
# 5. SA 음절·어절 hard retry + 유형별 안전한 출력 정규화
# ---------------------------------------------------------------------------


_COUNT_RE = re.compile(r"(\d+)\s*(음절|어절)")
_TOKEN_STATS_CACHE = {}


def parse_sa_length_constraint(question_text):
    matches = _COUNT_RE.findall(str(question_text))
    if len(matches) != 1:
        return None
    count_text, unit_kr = matches[0]
    target = int(count_text)
    if target < 1:
        return None
    return ("syllable" if unit_kr == "음절" else "eojeol", target)


def _token_stats(tokenizer):
    cache_key = id(tokenizer)
    if cache_key in _TOKEN_STATS_CACHE:
        return _TOKEN_STATS_CACHE[cache_key]
    vocab_size = len(tokenizer)
    syllables = torch.zeros(vocab_size, dtype=torch.long)
    chunks = torch.zeros(vocab_size, dtype=torch.long)
    starts_ws = torch.zeros(vocab_size, dtype=torch.bool)
    ends_ws = torch.zeros(vocab_size, dtype=torch.bool)
    all_ws = torch.zeros(vocab_size, dtype=torch.bool)
    for start in tqdm(range(0, vocab_size, 4096), desc="SA 길이 토큰 통계"):
        ids = list(range(start, min(start + 4096, vocab_size)))
        texts = tokenizer.batch_decode(
            [[token_id] for token_id in ids],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        for offset, text in enumerate(texts):
            if not text:
                continue
            token_id = start + offset
            syllables[token_id] = sum("가" <= char <= "힣" for char in text)
            chunks[token_id] = len(text.split())
            starts_ws[token_id] = text[0].isspace()
            ends_ws[token_id] = text[-1].isspace()
            all_ws[token_id] = not text.strip()
    value = {
        "syllables": syllables,
        "chunks": chunks,
        "starts_ws": starts_ws,
        "ends_ws": ends_ws,
        "all_ws": all_ws,
    }
    _TOKEN_STATS_CACHE[cache_key] = value
    return value


class KoreanLengthLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, spec, eos_token_id):
        self.spec = spec
        self.eos_token_id = int(eos_token_id)
        stats = _token_stats(tokenizer)
        self.syllables = stats["syllables"]
        self.chunks = stats["chunks"]
        self.starts_ws = stats["starts_ws"]
        self.ends_ws = stats["ends_ws"]
        self.all_ws = stats["all_ws"]
        self.prefix_length = None
        self.processed_length = 0
        self.word_count = 0
        self.at_boundary = True

    def _to(self, device):
        if self.syllables.device == device:
            return
        self.syllables = self.syllables.to(device)
        self.chunks = self.chunks.to(device)
        self.starts_ws = self.starts_ws.to(device)
        self.ends_ws = self.ends_ws.to(device)
        self.all_ws = self.all_ws.to(device)

    def __call__(self, input_ids, scores):
        self._to(scores.device)
        if self.prefix_length is None:
            self.prefix_length = input_ids.shape[1]
        generated = input_ids[0, self.prefix_length :]
        unit, target = self.spec
        if unit == "syllable":
            current = int(self.syllables[generated].sum().item())
            remaining = target - current
            if remaining <= 0:
                scores[0, :] = float("-inf")
                scores[0, self.eos_token_id] = 0.0
            else:
                scores[0, self.eos_token_id] = float("-inf")
                scores[0, self.syllables > remaining] = float("-inf")
            return scores

        for token_tensor in generated[self.processed_length :]:
            token_id = int(token_tensor.item())
            token_chunks = int(self.chunks[token_id].item())
            if token_chunks == 0:
                if bool(self.all_ws[token_id].item()):
                    self.at_boundary = True
                continue
            merge = self.word_count > 0 and not self.at_boundary and not bool(
                self.starts_ws[token_id].item()
            )
            self.word_count += token_chunks - 1 if merge else token_chunks
            self.at_boundary = bool(self.ends_ws[token_id].item())
        self.processed_length = len(generated)
        if self.word_count < target:
            scores[0, self.eos_token_id] = float("-inf")
        if self.word_count >= target:
            if self.word_count > 0 and not self.at_boundary:
                effective = torch.where(
                    self.starts_ws, self.chunks, self.chunks - 1
                )
            else:
                effective = self.chunks
            scores[0, effective > 0] = float("-inf")
        return scores


def _strip_thinking(text):
    text = str(text).strip()
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return re.sub(
        r"<think>.*?</think>", " ", text, flags=re.DOTALL | re.IGNORECASE
    ).strip()


def _normalize_mc(text):
    text = _strip_thinking(text)
    first = next((line.strip() for line in text.splitlines() if line.strip()), "")
    match = re.search(r"(?<!\d)([1-5](?:\s*[/,]\s*[1-5])*)(?!\d)", first)
    if not match:
        return "", False
    numbers = sorted(set(re.findall(r"[1-5]", match.group(1))), key=int)
    return "/".join(numbers), True


def _normalize_sa(text):
    text = _strip_thinking(text)
    text = re.sub(r"^(?:정답은|정답|답은|답|answer)\s*[:：]?\s*", "", text, flags=re.I)
    first = next((line.strip() for line in text.splitlines() if line.strip()), "")
    first = re.sub(r"\s+", " ", first).strip(" \t\"'“”‘’`")
    return first.rstrip("。.!?")


def _normalize_la(text):
    text = _strip_thinking(text)
    text = re.sub(r"^(?:정답|답변|answer)\s*[:：]?\s*", "", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip(" \t\"'“”‘’`")[:250].rstrip()


def _sa_requirements(question):
    return [
        (int(length), unit)
        for length, unit in re.findall(r"(\d+)\s*(음절|글자|어절)", str(question))
    ]


def _sa_length_match(answer, question):
    requirements = _sa_requirements(question)
    if not requirements:
        return None
    segments = [segment.strip() for segment in str(answer).split("/")]
    if len(requirements) > 1 and len(segments) != len(requirements):
        return False
    if len(requirements) == 1:
        segments = [str(answer).strip()]
    for segment, (target, unit) in zip(segments, requirements):
        actual = (
            len([token for token in segment.split() if token])
            if unit == "어절"
            else len(re.findall(r"[0-9A-Za-z가-힣]", segment))
        )
        if actual != target:
            return False
    return True


def _retry_feature(feature):
    copied = dict(feature)
    conversation = copy.deepcopy(feature["conversation"])
    for message in reversed(conversation):
        if message.get("role") == "user":
            message["content"] += (
                "\n\n이전 출력이 형식 또는 길이 조건을 지키지 못했습니다. "
                "정답을 다시 판단하고 출력 전에 음절·어절 수를 확인한 뒤 "
                "정답만 출력하세요."
            )
            break
    copied["conversation"] = conversation
    return copied


def _generate_once(
    model,
    processor,
    feature,
    max_length,
    dtype,
    *,
    retry=False,
    enforce_length=True,
):
    form = feature["question_form"]
    used_feature = _retry_feature(feature) if retry else feature
    batch = train_lora._move_batch_to_device(
        rag_infer.collate_generation_feature(processor, used_feature, max_length),
        train_lora._model_input_device(model),
    )
    kwargs = {
        "max_new_tokens": {"MC": 16, "SA": 32, "LA": 256}[form],
        "do_sample": False,
        "num_beams": 1,
        "use_cache": True,
        "pad_token_id": processor.tokenizer.pad_token_id,
        "eos_token_id": processor.tokenizer.eos_token_id,
    }
    spec = (
        parse_sa_length_constraint(feature.get("question", ""))
        if retry and enforce_length and form == "SA"
        else None
    )
    if spec is not None:
        kwargs["logits_processor"] = LogitsProcessorList(
            [
                KoreanLengthLogitsProcessor(
                    processor.tokenizer,
                    spec,
                    processor.tokenizer.eos_token_id,
                )
            ]
        )
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=dtype):
        generated = model.generate(**batch, **kwargs)
    return processor.batch_decode(
        generated.detach().cpu(),
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


def generate_champion_answer(model, processor, feature, max_length, dtype):
    form = feature["question_form"]
    primary_raw = _generate_once(
        model, processor, feature, max_length, dtype, retry=False
    )
    if form == "MC":
        primary, valid = _normalize_mc(primary_raw)
        if not valid:
            retry_raw = _generate_once(
                model, processor, feature, max_length, dtype, retry=True
            )
            retry, retry_valid = _normalize_mc(retry_raw)
            if retry_valid:
                return retry
        return primary if valid else "1"
    if form == "SA":
        primary = _normalize_sa(primary_raw)
        length_match = _sa_length_match(primary, feature.get("question", ""))
        if not primary or length_match is False:
            try:
                retry_raw = _generate_once(
                    model, processor, feature, max_length, dtype, retry=True
                )
            except (RuntimeError, TypeError, ValueError) as error:
                print("SA 길이 hard retry를 soft retry로 대체합니다:", error)
                retry_raw = _generate_once(
                    model,
                    processor,
                    feature,
                    max_length,
                    dtype,
                    retry=True,
                    enforce_length=False,
                )
            retry = _normalize_sa(retry_raw)
            retry_match = _sa_length_match(retry, feature.get("question", ""))
            if retry and retry_match is True:
                return retry
            if not primary and retry:
                return retry
        return primary or "확인 불가"
    return _normalize_la(primary_raw) or "확인 불가"


VALIDATION_EVAL_LOG = []
_EVAL_CALLS = Counter()


def champion_evaluate_generation(
    model,
    processor,
    dataset,
    batch_size,
    max_length,
    max_new_tokens,
    dtype,
):
    predictions = []
    references = []
    forms = []
    question_ids = []
    for index in tqdm(range(len(dataset)), desc="champion validation", leave=False):
        sample = dataset[index]
        predictions.append(
            generate_champion_answer(model, processor, sample, max_length, dtype)
        )
        references.append(str(sample["answer"]))
        forms.append(sample["question_form"])
        question_ids.append(sample.get("question_id", str(index)))
    metrics = aivqa_metrics.compute_vqa_metrics(predictions, references, forms)
    form = forms[0] if forms else "UNKNOWN"
    _EVAL_CALLS[form] += 1
    VALIDATION_EVAL_LOG.append(
        {
            "question_form": form,
            "epoch_call": _EVAL_CALLS[form],
            "metrics": metrics,
            "question_ids": question_ids,
            "predictions": predictions,
            "references": references,
        }
    )
    return predictions, metrics


def champion_generate_rag_predictions(
    model,
    processor,
    dataset,
    *,
    max_length,
    max_new_tokens,
    dtype,
    description,
):
    return [
        generate_champion_answer(model, processor, dataset[index], max_length, dtype)
        for index in tqdm(range(len(dataset)), desc=description, unit="sample")
    ]


type_train.evaluate_generation = champion_evaluate_generation
rag_pipeline.generate_rag_predictions = champion_generate_rag_predictions
rag_augmentation.generate_rag_predictions = champion_generate_rag_predictions


# SA는 더 오래 탐색하되 validation exact-match 기준 최고 epoch만 저장합니다.
_original_train_question_form = type_train.train_question_form


def train_question_form_with_budget(args, question_form, *positional, **keyword):
    local_args = copy.copy(args)
    local_args.epochs = TYPE_EPOCHS_BY_FORM[question_form]
    local_args.early_stopping_patience = TYPE_EPOCHS_BY_FORM[question_form]
    return _original_train_question_form(
        local_args, question_form, *positional, **keyword
    )


rag_pipeline.train_question_form = train_question_form_with_budget


print("Champion runtime patch 완료")
print("- Kanana revision/SDPA:", KANANA_REVISION)
print("- Selective RAG top-k:", RAG_TOP_K)
print("- 유형별 epochs:", TYPE_EPOCHS_BY_FORM)
print("- SA length mode: retry-only hard logits")

In [ ]:
print("적용된 핵심 함수 확인")
print("- retriever:", rag_infer.QdrantRetriever.retrieve.__name__)
print("- prompt:", rag_prompts.build_answer_feature.__name__)
print("- retrieval cache:", rag_pipeline.retrieve_dataset_candidates.__name__)
print("- validation:", type_train.evaluate_generation.__name__)
print("- test generation:", rag_pipeline.generate_rag_predictions.__name__)
print("- run dir:", RUN_DIR)

### 6. 전체 학습 및 추론

가장 오래 걸리는 셀입니다. 순서는 다음과 같습니다.

1. Base Kanana로 train+validation 검색어 생성 및 selective RAG cache 저장
2. Shared Adapter 2 epoch
3. MC 3 / SA 5 / LA 3 epoch, 매 epoch 유형별 검증 최고 Adapter 저장
4. Test 검색 및 유형별 최고 Adapter 추론
5. `answer.json` 생성

중단 후 다시 실행하면 완료된 검색 cache는 재사용됩니다. 학습 Adapter는
Drive의 고정 RUN_DIR에 저장됩니다.

In [ ]:
if not DO_TRAIN:
    raise ValueError(
        "이 노트북은 champion 전체 학습용입니다. DO_TRAIN=True로 실행하세요."
    )

pipeline_args = argparse.Namespace(
    model_id=MODEL_ID,
    train_json=json_paths["train"],
    validation_json=json_paths["validation"],
    test_json=json_paths["test"],
    dataset_root=DATASET_ROOT,
    output_dir=RUN_DIR.parent,
    qdrant_path=LOCAL_QDRANT_DIR,
    qdrant_url=None,
    collection=QDRANT_COLLECTION,
    model_cache=MODEL_CACHE_DIR,
    score_threshold=RETRIEVAL_BASE_THRESHOLD,
    retrieval_page_size=100,
    rag_device="cpu",
    rag_fp32=False,
    max_rag_chars=max(RAG_CONTEXT_LIMITS.values()),
    search_max_new_tokens=64,
    train_batch_size=1,
    eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    shared_learning_rate=SHARED_LEARNING_RATE,
    type_learning_rate=TYPE_LEARNING_RATE,
    shared_weight_decay=0.01,
    type_weight_decay=0.03,
    warmup_ratio=0.10,
    max_grad_norm=1.0,
    max_length=MAX_LENGTH,
    max_new_tokens=256,
    num_workers=0,
    seed=SEED,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    dtype=dtype_name,
    attn_implementation="sdpa",
    load_in_4bit=False,
    gradient_checkpointing=True,
)

answer_path = rag_pipeline.run_pipeline(pipeline_args)
print("전체 파이프라인 완료:", answer_path)

### 7. 유형별 최고 Validation 결과와 RAG 적용률 확인

In [ ]:
selection_keys = {
    "MC": "mc_accuracy", "SA": "sa_exact_match", "LA": "descriptive_avg"
}
best_logs = {}
for form, key in selection_keys.items():
    candidates = [
        row for row in VALIDATION_EVAL_LOG
        if row["question_form"] == form
    ]
    if not candidates:
        raise RuntimeError(f"{form} validation 기록이 없습니다.")
    best_logs[form] = max(candidates, key=lambda row: row["metrics"][key])

validation_rows = []
for form, row in best_logs.items():
    for question_id, prediction, reference in zip(
        row["question_ids"], row["predictions"], row["references"]
    ):
        validation_rows.append({
            "question_id": question_id,
            "question_form": form,
            "best_epoch_call": row["epoch_call"],
            "prediction": prediction,
            "reference": reference,
        })
validation_df = pd.DataFrame(validation_rows)
validation_path = RUN_DIR / "validation_best_predictions.csv"
validation_df.to_csv(validation_path, index=False, encoding="utf-8-sig")

mc = best_logs["MC"]["metrics"]["mc_accuracy"]
sa = best_logs["SA"]["metrics"]["sa_exact_match"]
descriptive = best_logs["LA"]["metrics"]["descriptive_avg"]
diagnostic_score = (mc + sa + descriptive) / 3
diagnostic = {
    "warning": (
        "Shared 단계가 train+validation을 사용하므로 완전한 홀드아웃 점수가 "
        "아니며, 리더보드 점수를 보장하지 않습니다."
    ),
    "best_epoch_call": {
        form: row["epoch_call"] for form, row in best_logs.items()
    },
    "mc_accuracy": mc,
    "sa_exact_match": sa,
    "descriptive_avg": descriptive,
    "diagnostic_score": diagnostic_score,
}
diagnostic_path = RUN_DIR / "validation_diagnostic.json"
diagnostic_path.write_text(
    json.dumps(diagnostic, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.DataFrame([
    {
        "metric": "accuracy", "value": mc * 100,
        "best_epoch": best_logs["MC"]["epoch_call"],
    },
    {
        "metric": "exact_match", "value": sa * 100,
        "best_epoch": best_logs["SA"]["epoch_call"],
    },
    {
        "metric": "descriptive_avg", "value": descriptive * 100,
        "best_epoch": best_logs["LA"]["epoch_call"],
    },
    {"metric": "diagnostic_score", "value": diagnostic_score * 100},
]))
print("검증 예측:", validation_path)
print("주의:", diagnostic["warning"])

In [ ]:
cache_rows = []
cache_dir = RUN_DIR / "rag_cache"
for cache_path in sorted(cache_dir.glob("*.json")):
    try:
        cached = json.loads(cache_path.read_text(encoding="utf-8"))
        rows = cached.get("rows", []) if isinstance(cached, dict) else cached
        applied = sum(bool(row.get("candidates")) for row in rows)
        reasons = Counter(
            candidate.get("payload", {}).get("_retrieval", {}).get("reason", "")
            for row in rows
            for candidate in row.get("candidates", [])
        )
        cache_rows.append({
            "cache": cache_path.name,
            "samples": len(rows),
            "rag_applied": applied,
            "apply_rate": applied / len(rows) if rows else 0.0,
            "reasons": dict(reasons),
        })
    except Exception as error:
        cache_rows.append({"cache": cache_path.name, "error": str(error)})
rag_usage_df = pd.DataFrame(cache_rows)
display(rag_usage_df)
rag_usage_path = RUN_DIR / "rag_usage_summary.csv"
rag_usage_df.to_csv(rag_usage_path, index=False, encoding="utf-8-sig")

### 8. 제출 JSON·ZIP 및 실험 요약 생성

In [ ]:
submission = json.loads(Path(answer_path).read_text(encoding="utf-8"))
test_records = records["test"]
assert isinstance(submission, list) and len(submission) == 800
assert len(test_records) == len(submission)
for source, predicted in zip(test_records, submission):
    assert str(source["metadata"]["question_id"]) == str(
        predicted["metadata"]["question_id"]
    )
    answer = predicted.get("model_output", {}).get("answer")
    assert isinstance(answer, str) and answer.strip()

submission_path = RUN_DIR / "submission_kanana15v_3b_unified_rag_champion.json"
submission_path.write_text(
    json.dumps(submission, ensure_ascii=False, indent=2), encoding="utf-8"
)
submission_zip = RUN_DIR / "submission_kanana15v_3b_unified_rag_champion.zip"
with zipfile.ZipFile(submission_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(submission_path, arcname=submission_path.name)

experiment_summary = {
    "run_name": RUN_NAME,
    "repo_url": REPO_URL,
    "repo_commit": REPO_COMMIT,
    "model_id": MODEL_ID,
    "model_revision": KANANA_REVISION,
    "rag_documents": len(normalized_rows),
    "rag_sha256": RAG_SHA256,
    "rag_sources": dict(source_counts),
    "rag_image_coverage": coverage,
    "rag_build_mode": RAG_BUILD_MODE,
    "retrieval_signature": RETRIEVAL_SIGNATURE,
    "shared_epochs": SHARED_EPOCHS,
    "type_epochs": TYPE_EPOCHS_BY_FORM,
    "learning_rates": {
        "shared": SHARED_LEARNING_RATE,
        "typed": TYPE_LEARNING_RATE,
    },
    "sa_length_mode": "free_generation_then_retry_only_hard_constraint",
    "validation_diagnostic": diagnostic,
    "submission_json": str(submission_path),
    "submission_zip": str(submission_zip),
}
summary_path = RUN_DIR / "champion_experiment_summary.json"
summary_path.write_text(
    json.dumps(experiment_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("제출할 파일:", submission_path)
print("편의용 ZIP:", submission_zip)
print("실험 요약:", summary_path)

### 결과 확인 순서

1. `rag_usage_summary.csv`에서 RAG가 전체 문항에 강제되지 않았는지 확인
2. `validation_diagnostic.json`에서 MC / SA / LA 최고 epoch 확인
3. `validation_best_predictions.csv`에서 특히 SA 오답과 길이 준수 확인
4. 리더보드에는 **JSON 파일만** 제출
   - `submission_kanana15v_3b_unified_rag_champion.json`
5. 기존 최고점과 지표별 비교
   - 전체 44.8395
   - accuracy 73.7349
   - exact_match 25.7282
   - descriptive_avg 35.0555

첫 결과가 47점에 못 미치더라도, 이 실행의 retrieval cache와 유형별
validation 결과를 이용해 임계값·RAG 적용 대상만 바꾼 다음 실험을 설계할 수
있습니다. 여러 설정을 동시에 바꾸지 말고 리더보드 비교 실험을 하나씩
누적하세요.